# backprop-pop-outgrad-loop — faded example 2: Complete the leaf-grad accumulation branch

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `backprop-pop-outgrad-loop`. The last cell reports your progress on the `Backprop: backprop pop-outgrad loop` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper.
    `.grad` accumulates the leaf gradient at the end of the reverse pass."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: backprop pop-outgrad loop` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`backprop-pop-outgrad-loop`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "backprop-pop-outgrad-loop"
DD_SUBTOPIC = "Backprop: backprop pop-outgrad loop"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

When the driver pops a **leaf** (`recipe is None`), it must write the popped grad into the leaf's `.grad`, accumulating with `+` if `.grad` already holds a value from an earlier consumer. Overwriting instead of accumulating loses gradient when a leaf feeds multiple downstream ops. Non-leaf nodes never touch `.grad`; their grad stays in the `grads` dict and routes onward.

## Faded exercise 2

### Faded — finish the leaf `.grad` write

The driver below handles a graph `out = (w*a) + (w*b)` where leaf `w` is consumed by two separate `mul` ops, so `w` accumulates grad from two reverse steps. Complete the leaf branch so `.grad` **accumulates** (writes the first time, adds thereafter). Correct behaviour gives `w.grad == a + b`.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
import numpy as np
import torch as t
from einops import rearrange, reduce, repeat
np.random.seed(0); t.manual_seed(0)

class Recipe:
    def __init__(self, func, args, kwargs, parents):
        self.func = func; self.args = args; self.kwargs = kwargs; self.parents = parents

class MiniTensor:
    def __init__(self, array, recipe=None):
        self.array = array; self.recipe = recipe; self.grad = None

def _mul_back0(go, out, x, y):  return go * y
def _mul_back1(go, out, x, y):  return go * x

def backprop(end_node, end_grad, sorted_graph, back_funcs) -> None:
    grads = {id(end_node): end_grad}
    for node in sorted_graph:
        nid = id(node)
        if nid not in grads:
            continue
        grad_out = grads.pop(nid)
        if node.recipe is None:
            node.grad = grad_out if node.grad is None else node.grad + grad_out
            continue
        for argnum, parent in node.recipe.parents.items():
            bf = back_funcs[(node.recipe.func, argnum)]
            gp = bf(grad_out, node.array, *node.recipe.args, **node.recipe.kwargs)
            pid = id(parent)
            grads[pid] = grads.get(pid, 0) + gp

def _add_back0(go, out, x, y):  return go
def _add_back1(go, out, x, y):  return go

t.manual_seed(0)
w = MiniTensor(t.randn(3)); a = MiniTensor(t.randn(3)); b = MiniTensor(t.randn(3))
p_arr = w.array * a.array
p = MiniTensor(p_arr, Recipe('mul', (w.array, a.array), {}, {0: w, 1: a}))
q_arr = w.array * b.array
q = MiniTensor(q_arr, Recipe('mul', (w.array, b.array), {}, {0: w, 1: b}))
out_arr = p_arr + q_arr
out = MiniTensor(out_arr, Recipe('add', (p_arr, q_arr), {}, {0: p, 1: q}))
back_funcs = {('mul', 0): _mul_back0, ('mul', 1): _mul_back1,
              ('add', 0): _add_back0, ('add', 1): _add_back1}
backprop(out, t.ones_like(out.array), [out, p, q, w, a, b], back_funcs)


def _test():
    assert w.grad is not None, 'w.grad was never set'
    expected = a.array + b.array
    assert t.allclose(w.grad, expected), f'expected a+b, got {w.grad}'
    assert t.allclose(a.grad, w.array), 'a.grad should be w'
    assert t.allclose(b.grad, w.array), 'b.grad should be w'


try:
    _test()
    _dd_passed.add('faded2')
    print('[Delta Drills] faded2 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import numpy as np
import torch as t
from einops import rearrange, reduce, repeat
np.random.seed(0); t.manual_seed(0)

class Recipe:
    def __init__(self, func, args, kwargs, parents):
        self.func = func; self.args = args; self.kwargs = kwargs; self.parents = parents

class MiniTensor:
    def __init__(self, array, recipe=None):
        self.array = array; self.recipe = recipe; self.grad = None

def _mul_back0(go, out, x, y):  return go * y
def _mul_back1(go, out, x, y):  return go * x

def backprop(end_node, end_grad, sorted_graph, back_funcs) -> None:
    grads = {id(end_node): end_grad}
    for node in sorted_graph:
        nid = id(node)
        if nid not in grads:
            continue
        grad_out = grads.pop(nid)
        if node.recipe is None:
            node.grad = grad_out if node.grad is None else node.grad + grad_out
            continue
        for argnum, parent in node.recipe.parents.items():
            bf = back_funcs[(node.recipe.func, argnum)]
            gp = bf(grad_out, node.array, *node.recipe.args, **node.recipe.kwargs)
            pid = id(parent)
            grads[pid] = grads.get(pid, 0) + gp

def _add_back0(go, out, x, y):  return go
def _add_back1(go, out, x, y):  return go

t.manual_seed(0)
w = MiniTensor(t.randn(3)); a = MiniTensor(t.randn(3)); b = MiniTensor(t.randn(3))
p_arr = w.array * a.array
p = MiniTensor(p_arr, Recipe('mul', (w.array, a.array), {}, {0: w, 1: a}))
q_arr = w.array * b.array
q = MiniTensor(q_arr, Recipe('mul', (w.array, b.array), {}, {0: w, 1: b}))
out_arr = p_arr + q_arr
out = MiniTensor(out_arr, Recipe('add', (p_arr, q_arr), {}, {0: p, 1: q}))
back_funcs = {('mul', 0): _mul_back0, ('mul', 1): _mul_back1,
              ('add', 0): _add_back0, ('add', 1): _add_back1}
backprop(out, t.ones_like(out.array), [out, p, q, w, a, b], back_funcs)
```
</details>